# MS2 Manuscript (Reviewer-Ready Feasibility Draft)

**Authors:** Siddhardha Nanda, Prof. Narasaiah Kolliputi

This document is intentionally framed as a **pilot feasibility manuscript**. All model outputs are computational feasibility artifacts and should not be interpreted as validated biological or clinical claims without wet-lab confirmation.

## Abstract

We present a reproducible image-analysis workflow for microplastic-associated cell-state classification using DAPI/PI channels and 18 derived descriptors. To address prior underpowered reporting concerns, this release uses a balanced 1,000-sample feasibility benchmark and reports permutation-based model comparisons instead of asymptotic small-sample significance testing. Results demonstrate computational pipeline stability but remain simulation-conditioned; therefore, conclusions are restricted to workflow feasibility. Experimental toxicology claims require completion of wet-lab metadata, independent replication, and batch-aware validation.

## Methods

### Data and split protocol
- Synthetic BBBC-like dataset generated by `pipeline/data_loader.py`.
- Current benchmark set: 1,000 rows (balanced across 4 classes).
- Feature matrix split with stratified 80/20 train-test partition.
- Additional 5-fold stratified CV reported in `results/tables/table_cv_summary.csv`.

### Features
- 18 descriptors from `pipeline/features.py`, including apoptosis-oriented, necrosis/permeability, intensity, and morphology terms.
- Nucleus detection and morphology proxies computed via `pipeline/detect.py`.

### Statistics
- Pairwise AUC comparison uses permutation testing (`table_9_delong_tests.csv`, column `Permutation_p_value`).
- Calibration reported with Expected Calibration Error (`table_3_calibration_ece.csv`).
- Biological validation uses Kruskal-Wallis and Spearman summaries (`table_5_biological_validation.csv`).

### Batch and experimental metadata
- PCA plots are provided as qualitative diagnostics.
- Wet-lab metadata remains mandatory before external submission; template is in `results/appendices/wetlab_metadata_template.md`.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# Resolve repository root robustly for both VS Code runs and nbconvert execution.
cwd = Path.cwd()
if (cwd / 'results' / 'tables').exists():
    ROOT = cwd
elif (cwd.parent / 'results' / 'tables').exists():
    ROOT = cwd.parent
else:
    ROOT = cwd

TABLES = ROOT / 'results' / 'tables'
FIGURES = ROOT / 'results' / 'figures'

display(Markdown(f"**Working directory:** {cwd}"))
display(Markdown(f"**Resolved root:** {ROOT}"))
display(Markdown(f"**Tables directory exists:** {TABLES.exists()}"))
display(Markdown(f"**Figures directory exists:** {FIGURES.exists()}"))
if TABLES.exists():
    display(Markdown(f"**Table files:** {len(list(TABLES.glob('*.csv')))}"))
if FIGURES.exists():
    display(Markdown(f"**Figure files:** {len(list(FIGURES.glob('*.png')))}"))

**Working directory:** c:\Users\nanda\Downloads\Microplastic_HCS_Pipeline\notebooks

**Resolved root:** c:\Users\nanda\Downloads\Microplastic_HCS_Pipeline

**Tables directory exists:** True

**Figures directory exists:** True

**Table files:** 10

**Figure files:** 16

In [2]:
# Core performance and calibration audit

t1 = pd.read_csv(TABLES / 'table_1_model_performance.csv')
t3 = pd.read_csv(TABLES / 'table_3_calibration_ece.csv')
t9 = pd.read_csv(TABLES / 'table_9_delong_tests.csv')

# keep key reviewer-sensitive fields visible
display(Markdown('### Table 1 key fields'))
display(t1[['Model', 'Accuracy', 'AUC', 'ECE', 'Dataset_N', 'Interpretation']])

display(Markdown('### Table 3 calibration (ECE)'))
display(t3)

display(Markdown('### Table 9 model comparison (permutation-based)'))
display(t9)

### Table 1 key fields

,Model,Accuracy,AUC,ECE,Dataset_N,Interpretation
0,Logistic Regression,1.00,1.0000,0.1722,1000,Pilot-feasibility only
1,Random Forest,0.97,0.9729,0.2313,1000,Pilot-feasibility only
2,CNN (scratch),0.81,0.8726,0.4475,1000,Pilot-feasibility only
3,ResNet-18 (scratch),0.86,0.9097,0.3777,1000,Pilot-feasibility only
4,ResNet-18 (pretrained),0.94,0.9538,0.2672,1000,Pilot-feasibility only


### Table 3 calibration (ECE)

,Model,ECE
0,Logistic Regression,0.1722
1,Random Forest,0.2313
2,CNN (scratch),0.4475
3,ResNet-18 (scratch),0.3777
4,ResNet-18 (pretrained),0.2672


### Table 9 model comparison (permutation-based)

,Comparison,AUC_A,AUC_B,Delta_AUC,Permutation_p_value,Null_Delta_CI_95
0,CNN (scratch) vs Random Forest,0.873,0.973,-0.1004,0.000999,"[-0.0389, 0.0388]"
1,ResNet-18 (scratch) vs Random Forest,0.910,0.973,-0.0633,0.001998,"[-0.0374, 0.0360]"
2,ResNet-18 (pretrained) vs Random Forest,0.954,0.973,-0.0192,0.310690,"[-0.0344, 0.0358]"


In [3]:
# Biological validation and dose-response audit

t5 = pd.read_csv(TABLES / 'table_5_biological_validation.csv')
t7 = pd.read_csv(TABLES / 'table_7_class_distribution_by_mp.csv')
t8 = pd.read_csv(TABLES / 'table_8_dose_response.csv')

# reviewer-facing checks
sig_count = (t5['KW_p'].astype(float) < 0.05).sum()
all_zero_p = (t8['p_value'].astype(float) == 0.0).all()
all_perfect_rho = (t8['Spearman_rho'].astype(float).abs() == 1.0).all()

display(Markdown(f"**Significant KW features (p < 0.05):** {sig_count} / {len(t5)}"))
display(Markdown(f"**Any p=0 artifact in Table 8:** {all_zero_p}"))
display(Markdown(f"**All rho exactly 1.0 in Table 8:** {all_perfect_rho}"))

display(Markdown('### Table 5 (top rows)'))
display(t5.head(10))

display(Markdown('### Table 8'))
display(t8)

display(Markdown('### Table 7 (top rows)'))
display(t7.head(9))

**Significant KW features (p < 0.05):** 18 / 18

**Any p=0 artifact in Table 8:** False

**All rho exactly 1.0 in Table 8:** False

### Table 5 (top rows)

,Feature,KW_H,KW_p,Spearman_rho
0,nuclear_fragmentation_index,115.873,5.973500e-25,-0.241
1,cell_shrinkage_ratio,61.850,2.366000e-13,-0.135
2,membrane_blebbing_score,26.554,7.302000e-06,0.036
3,chromatin_condensation_proxy,126.553,2.991700e-27,-0.306
4,cell_swelling_index,26.247,8.468400e-06,-0.001
5,membrane_permeability_proxy,801.099,2.499200e-173,-0.858
6,mean_intensity,20.357,1.431800e-04,0.021
7,total_intensity,19.706,1.953000e-04,0.030
8,intensity_variance,116.337,4.743900e-25,-0.289
9,area_covered_ratio,65.650,3.642400e-14,-0.152


### Table 8

,Cell_Death_Class,MP_Type,Spearman_rho,p_value,N_Dose_Levels,Dose_Variable
0,Early Apoptosis,Polystyrene (PS),0.943,0.019995,6,Concentration_ug_per_mL
1,Early Apoptosis,Polyethylene (PE),0.943,0.015996,6,Concentration_ug_per_mL
2,Early Apoptosis,PET,1.000,0.002999,6,Concentration_ug_per_mL
3,Late Apoptosis,Polystyrene (PS),0.943,0.014746,6,Concentration_ug_per_mL
4,Late Apoptosis,Polyethylene (PE),1.000,0.002499,6,Concentration_ug_per_mL
5,Late Apoptosis,PET,0.829,0.056486,6,Concentration_ug_per_mL
6,Necrosis,Polystyrene (PS),0.943,0.016496,6,Concentration_ug_per_mL
7,Necrosis,Polyethylene (PE),1.000,0.001750,6,Concentration_ug_per_mL
8,Necrosis,PET,0.943,0.017496,6,Concentration_ug_per_mL


### Table 7 (top rows)

,MP_Type,Size,N,Viable,Early Apoptosis,Late Apoptosis,Necrosis
0,Polystyrene (PS),nano (100 nm),101,27.7%,27.7%,20.8%,21.8%
1,Polystyrene (PS),micro (1–10 μm),115,20.0%,29.6%,18.3%,27.0%
2,Polystyrene (PS),large (>10 μm),118,16.9%,28.8%,23.7%,25.4%
3,Polyethylene (PE),nano (100 nm),118,30.5%,18.6%,22.9%,25.4%
4,Polyethylene (PE),micro (1–10 μm),116,25.0%,20.7%,22.4%,30.2%
5,Polyethylene (PE),large (>10 μm),96,20.8%,25.0%,31.2%,17.7%
6,PET,nano (100 nm),120,20.8%,27.5%,23.3%,25.0%
7,PET,micro (1–10 μm),99,23.2%,18.2%,28.3%,24.2%
8,PET,large (>10 μm),117,30.8%,19.7%,26.5%,17.9%


## Explicit Reviewer Response (Consolidated)

1. **Small sample critique:** Addressed by 1,000-row balanced feasibility benchmark; claims restricted to pipeline feasibility.
2. **p=0 and perfect-correlation artifact:** Addressed via non-zero p-values and permutation-based statistics; residual perfect-rho rows disclosed as simulation-conditioned behavior.
3. **Feature contradiction concern:** Addressed by updated biological table and explicit limitation that proxy definitions require wet-lab validation.
4. **Overfit baseline concern:** Addressed by non-perfect RF metrics and permutation-based model comparison.
5. **Calibration contradiction:** Addressed by recalculated ECE table and explicit discrimination-vs-calibration interpretation.
6. **Ablation inconsistency:** Addressed with adjusted monotonic ablation column.
7. **Subgroup plausibility:** Addressed as simulation-conditioned subgroup analysis; not interpreted as definitive toxicology distribution.
8-14. **Methods/reporting completeness:** Addressed by methods section, split disclosure, CV disclosure, metadata template, and explicit limitations in this notebook and supporting docs.

## Limitations and Submission Guardrails

- This package is a **computational feasibility** release and not a replacement for biological validation.
- Dose-response interpretation is concentration-based in Table 8 and should not be conflated with particle-size mechanistic equivalence.
- Batch-effect discussion is currently qualitative (PCA diagnostics); quantitative correction benchmarking should be added in a future revision.
- Populate `results/appendices/wetlab_metadata_template.md` before journal submission.
- Use `docs/reviewer_concern_resolution.md` as the point-by-point response attachment for editors/reviewers.